In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import OneHotEncoder,MultiLabelBinarizer,LabelEncoder,MinMaxScaler,StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import ADASYN
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.utils import class_weight
from sklearn.metrics import classification_report
import numpy as np

In [2]:
data=pd.read_csv('Lending_Loan_Company_Data_Analysis.csv')
data

,Credit_Policy,Loan_Purpose,Interest_Rate,Monthly_Installment,Natural_log_Annual_Income,Debt_Income_Ratio,FICO_Score,Days_with_Credit_Line,Revolving_Balance,Revolving_Utilization,Enquiries_Last_6_Months,Deliquency_Last_2_Yrs,Public_Derogatory_Records,Paid_Status
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,Paid
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,Paid
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,Paid
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,Paid
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9573,0,all_other,0.1461,344.76,12.180755,10.39,672,10474.000000,215372,82.1,2,0,0,Unpaid
9574,0,all_other,0.1253,257.70,11.141862,0.21,722,4380.000000,184,1.1,5,0,0,Unpaid
9575,0,debt_consolidation,0.1071,97.81,10.596635,13.09,687,3450.041667,10036,82.9,8,0,0,Unpaid
9576,0,home_improvement,0.1600,351.58,10.819778,19.18,692,1800.000000,0,3.2,5,0,0,Unpaid


# Feature Engineering

## Check for Collinearity B/W the Columns

In [54]:
for i in data_num.columns:
    corr=data_num.corr()
    corr[i].sort_values(ascending=False)
    print(corr)

                           Credit_Policy  Interest_Rate  Monthly_Installment  \
Credit_Policy                   1.000000      -0.294089             0.058770   
Interest_Rate                  -0.294089       1.000000             0.276140   
Monthly_Installment             0.058770       0.276140             1.000000   
Natural_log_Annual_Income       0.034906       0.056383             0.448102   
Debt_Income_Ratio              -0.090901       0.220006             0.050202   
FICO_Score                      0.348319      -0.714821             0.086039   
Days_with_Credit_Line           0.099026      -0.124022             0.183297   
Revolving_Balance              -0.187518       0.092527             0.233625   
Revolving_Utilization          -0.104095       0.464837             0.081356   
Enquiries_Last_6_Months        -0.535511       0.202780            -0.010419   
Deliquency_Last_2_Yrs          -0.076318       0.156079            -0.004368   
Public_Derogatory_Records      -0.054243

### Since, Most of the Columns have the Scores less Than 0.5 between them, There is no need to drop off the columns

# Model Building

## 1. Encode the Labels

In [58]:
data_model=data.copy()

In [59]:
le=LabelEncoder()

In [60]:
data_model

,Credit_Policy,Loan_Purpose,Interest_Rate,Monthly_Installment,Natural_log_Annual_Income,Debt_Income_Ratio,FICO_Score,Days_with_Credit_Line,Revolving_Balance,Revolving_Utilization,Enquiries_Last_6_Months,Deliquency_Last_2_Yrs,Public_Derogatory_Records,Paid_Status
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,Paid
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,Paid
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,Paid
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,Paid
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9573,0,all_other,0.1461,344.76,12.180755,10.39,672,10474.000000,215372,82.1,2,0,0,Unpaid
9574,0,all_other,0.1253,257.70,11.141862,0.21,722,4380.000000,184,1.1,5,0,0,Unpaid
9575,0,debt_consolidation,0.1071,97.81,10.596635,13.09,687,3450.041667,10036,82.9,8,0,0,Unpaid
9576,0,home_improvement,0.1600,351.58,10.819778,19.18,692,1800.000000,0,3.2,5,0,0,Unpaid


In [61]:
data['Loan_Purpose'].unique()

array(['debt_consolidation', 'credit_card', 'all_other',
       'home_improvement', 'small_business', 'major_purchase',
       'educational'], dtype=object)

In [62]:
data_model['Loan_Purpose'].unique()

array(['debt_consolidation', 'credit_card', 'all_other',
       'home_improvement', 'small_business', 'major_purchase',
       'educational'], dtype=object)

In [63]:
data

,Credit_Policy,Loan_Purpose,Interest_Rate,Monthly_Installment,Natural_log_Annual_Income,Debt_Income_Ratio,FICO_Score,Days_with_Credit_Line,Revolving_Balance,Revolving_Utilization,Enquiries_Last_6_Months,Deliquency_Last_2_Yrs,Public_Derogatory_Records,Paid_Status
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,Paid
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,Paid
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,Paid
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,Paid
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9573,0,all_other,0.1461,344.76,12.180755,10.39,672,10474.000000,215372,82.1,2,0,0,Unpaid
9574,0,all_other,0.1253,257.70,11.141862,0.21,722,4380.000000,184,1.1,5,0,0,Unpaid
9575,0,debt_consolidation,0.1071,97.81,10.596635,13.09,687,3450.041667,10036,82.9,8,0,0,Unpaid
9576,0,home_improvement,0.1600,351.58,10.819778,19.18,692,1800.000000,0,3.2,5,0,0,Unpaid


## Description of Labels

### 1. Loan Purpose
#### * debt_consolidation - 2
#### * credit_card - 1
#### * all_other - 0
#### * home_improvement -4
#### * small_business - 6
#### * major_purchase - 5
#### * educational - 3

### 2. Paid Status
#### * Paid - 0
#### * Unpaid - 1

In [65]:
data_model['Installment_to_Income'] = data_model['Monthly_Installment'] / (data['Natural_log_Annual_Income'] / 12)
data_model['Interest_Installment'] = data_model['Interest_Rate'] * data_model['Monthly_Installment']
data_model['Revolving_to_Income'] = data_model['Revolving_Balance'] / data_model['Natural_log_Annual_Income']
data_model['Credit_Length_Years'] = data_model['Days_with_Credit_Line'] / 365
data_model['Risk_Score'] = data_model['Interest_Rate'] / data_model['FICO_Score']

In [66]:
x=data_model.drop(['Paid_Status','Natural_log_Annual_Income'],axis=1)
y=le.fit_transform(data_model['Paid_Status'])

In [67]:
xtrain,xtemp,ytrain,ytemp=train_test_split(x,y,test_size=0.3,random_state=42)
xeval,xtest,yeval,ytest=train_test_split(xtemp,ytemp,test_size=0.5,random_state=42)

In [68]:

cat_cols = ['Loan_Purpose','Credit_Policy', 'Loan_Purpose', 'Enquiries_Last_6_Months' ,'Deliquency_Last_2_Yrs','Public_Derogatory_Records']

# Outlier numeric
robust_cols = [
    'Interest_Rate',
    'Monthly_Installment',
    'FICO_Score',
    'Days_with_Credit_Line',
    'Revolving_Balance',
    'Installment_to_Income',
    'Interest_Installment'
    
]

# Remaining numeric
standard_cols = [col for col in xtrain.columns 
                 if col not in robust_cols + cat_cols]



In [69]:

preprocessor = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), robust_cols),
        ('standard', StandardScaler(), standard_cols),
        ('cat',OneHotEncoder(handle_unknown='ignore'), cat_cols)   
    ]
)

# Step 1: Fit scaler on original train
X_train_scaled = preprocessor.fit_transform(xtrain)

# Step 2: Apply same transform
X_eval = preprocessor.transform(xeval)
X_test = preprocessor.transform(xtest)


In [164]:

class_weight = {0:1, 1:3}


In [166]:
model = Sequential()

# Input Layer + Hidden Layer 1
model.add(Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dropout(0.3))

# Hidden Layer 2
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

# Hidden Layer 3
model.add(Dense(16, activation='relu'))

# Output Layer
model.add(Dense(1, activation='sigmoid'))  # Binary classification

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['Precision', 'Recall']
)

# Early Stopping (VERY IMPORTANT)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Train
history = model.fit(
    X_train_scaled, ytrain,
    validation_data=(X_eval, yeval),
    class_weight=class_weight,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop]
    
)

Epoch 1/200
210/210 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - Precision: 0.1846 - Recall: 0.3314 - loss: 0.8801 - val_Precision: 0.2680 - val_Recall: 0.2213 - val_loss: 0.5420
Epoch 2/200
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.2612 - Recall: 0.1880 - loss: 0.8174 - val_Precision: 0.3096 - val_Recall: 0.2596 - val_loss: 0.5300
Epoch 3/200
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.2921 - Recall: 0.2217 - loss: 0.8254 - val_Precision: 0.3402 - val_Recall: 0.2809 - val_loss: 0.5102
Epoch 4/200
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.3064 - Recall: 0.2435 - loss: 0.7921 - val_Precision: 0.3313 - val_Recall: 0.2298 - val_loss: 0.5291
Epoch 5/200
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.3424 - Recall: 0.2879 - loss: 0.7981 - val_Precision: 0.3318 - val_Recall: 0.3149 - val_loss: 0.5224
Epoch 6/200
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - Precision: 0.3025 - Recall: 0.2621 - loss: 0.8269 - val_Precision: 0.3300 - val_Recall: 0.2851 - val

In [167]:
loss, precision, recall = model.evaluate(X_eval, yeval)
print("Loss:", loss)
print("Precision:", precision)
print("Recall:", recall)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - Precision: 0.3205 - Recall: 0.1738 - loss: 0.4939
Loss: 0.48141801357269287
Precision: 0.29411765933036804
Recall: 0.1489361673593521


In [196]:
ypred=model.predict(X_eval)

ypred_t = (model.predict(X_eval) > 0.45).astype(int)

print(classification_report(yeval, ypred_t))


45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1202
           1       0.29      0.22      0.25       235

    accuracy                           0.78      1437
   macro avg       0.57      0.56      0.56      1437
weighted avg       0.76      0.78      0.77      1437



In [197]:
ypred_Test=model.predict(X_test)

ypred_test = (model.predict(X_test) > 0.45).astype(int)

print(classification_report(ytest, ypred_test))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

           0       0.87      0.89      0.88      1206
           1       0.33      0.29      0.31       231

    accuracy                           0.79      1437
   macro avg       0.60      0.59      0.60      1437
weighted avg       0.78      0.79      0.79      1437



#                                      Insights

1. The dataset exhibited class imbalance with fewer 'Unpaid' (defaulter) cases compared to 'Paid'. 
Initially, ADASYN was used to generate synthetic minority samples, which improved recall but led to a high number of false positives (low precision).

To address this, class weights were introduced during model training. This approach improved the balance between precision and recall by penalizing misclassification of defaulters more heavily, without introducing synthetic noise.

2. A comparison between ADASYN, class weights, and their combination revealed that:

- ADASYN alone increased recall significantly but reduced precision due to over-prediction of the minority class.
- Class weights provided a more stable and balanced performance, improving precision while maintaining reasonable recall.
- Combining ADASYN with class weights led to overcompensation toward the minority class, resulting in excessive false positives and degraded overall performance.

Hence, class weights alone were selected as the optimal strategy for handling class imbalance.

3. The default classification threshold of 0.5 resulted in very low recall for the minority class (Unpaid), indicating that the model failed to identify a significant portion of defaulters.

To address this, threshold tuning was performed:

- Lower thresholds (e.g., 0.3) increased recall but caused a sharp drop in precision due to excessive false positives.
- Higher thresholds (e.g., 0.6–0.7) improved precision but failed to detect defaulters (very low recall).
- A threshold of 0.45 provided the best trade-off, achieving a balanced performance between precision and recall.

This demonstrates the importance of threshold tuning in imbalanced classification problems.

4. The final model, trained using class weights and evaluated at a threshold of 0.45, achieved:

- Recall ~33% for the 'Unpaid' class, indicating the model's ability to detect nearly half of potential defaulters.
- Precision ~29%, suggesting moderate false positives, which is acceptable in risk-sensitive applications.
- F1-score ~0.31, reflecting a balanced trade-off between precision and recall.

Additionally, consistent performance across evaluation and test datasets indicates good generalization capability.


# Summary

From a business perspective, the model prioritizes the identification of defaulters (Unpaid customers), as missing such cases can lead to financial losses.

By lowering the decision threshold to 0.4, the model adopts a slightly aggressive strategy, flagging customers with moderate risk levels. While this increases false positives, it significantly reduces the risk of approving high-risk applicants.

This trade-off aligns with real-world financial decision-making, where minimizing default risk is often more critical than maximizing approval rates.
